# Datamine ATTSET Process Example

This notebook demonstrates how to configure and run the **`attset`** process wrapper in `dmstudio`.

## Process Description

## ATTSET

Set the values of nominated attribute fields in a file based on the values of a given data field in the input file. Attribute values are defined relative to data ranges or by matching patterns.

The &**LEGEND** file can contain multiple attribute definitions. A legend name provides the key for each definition. Each legend may contain definitions for several data fields. For each data field, one or more attribute fields may be defined.

Each default field name in &**LEGEND** may be redefined on the command line via symbolic field names in the expected way. e.g. * **DATFLD1**(NORTH).

Where data and/or attribute fields are supplied in the legend definition file, the values specified on the command line must exist in the file.

Where the attribute field does not exist in the input file, but will be created in the output file, **ATTSET** will determine the type and length from the contents of the **ATTVALUE** field values.

To simplify the input and maintenance of the legend definition file, the specification of ranges does not differentiate between numeric and alphanumeric fields - all values are entered with alphanumeric fields. The following rules apply for the entry of range information to handle discrete, continuous and non-continuous data :-

The values entered in **DATMIN** , **DATMAX** and **DATEXP** must match the data type of the data field. Alphanumeric data entered into the **DATMIN** and **DATMAX** fields are handled like the existing retrieval criteria. Special numeric values Tr,Dl,+,- are recognized for numeric data.

Data entered into the **DATEXP** field is handled in the same way as the **PICREC** process. If full regular expressions are required, then **REGEXP** should precede the full regular expression.

If only **DATMIN** is defined, discrete values are assumed.

If some **DATMIN** or **DATMAX** values are missing an error will be reported. The standard upper and lower limit values for numeric and alphanumeric data should be used where no specific upper or lower bound is to be specified.

If both **DATMIN** and **DATMAX** are defined, DATMIN is inclusive on the lower bound, and non-inclusive on the upper bound. If DATMIN=DATMAX then a test for discrete values is assumed.

Overlapping ranges are not allowed. If, for example, a background color is to be set and a limited number of other ranges specified, then each interval must be specified.

During data entry in **[AED](<aed.md>)** or a similar input process, blank (space) field values can be left in the file, to eliminate the need to re-enter repeated field values. An additional process, **ATTCHK** is provided to fill in blank field values with the previous defined field value (in the column), and validate the content of the legend file.

**ATTSET** provides the functionality of multi-field **[SETVAL](<setval.md>)** or **[GENTRA](<gentra.md>)** but oriented to customized menu applications.

### Input Files:

* **in** (Input data file):
  Overwritten
  Required=No

* **legend** (Undefined):
  Legend definition file. The following standard field names are expected: LEGEND A8
  Legend key. DATFIELD A8 Data field in input file. DATMIN A12 Minimum value. DATMAX A12
  Maximum value. DATEXP A40 Match regular expression. ATTFIELD A8 Attribute field name.
  ATTVALUE A12 Attribute field value. Alternate field names can be supplied to the process
  by specification through the symbolic field names.
  Required=Yes

### Output Files:

* **out** (Undefined):
  Output file containing the additional attribute fields. May be the same as the input
  file but additional attribute fields will be ignored.
  Required=Yes

### Fields:

* **datflds** (Undefined : Undefined):
  Optional data field. If no DATFIELD field is supplied, then DATFLD1 is used to specify the single required data field. Otherwise DATFLD1..5 can be used to select a subset of the data fields.
  Default=Undefined
  Required=No

* **attribs** (Undefined : Undefined):
  First optional attribute field. If no ATTFIELD field is specified, then ATTRIB1 is used to specify the single required attribute field. Otherwise ATTRIB1..5 can be used to select a subset of the attribute fields.
  Default=Undefined
  Required=No

### Parameters:

* **mode**:
  Type of validation to be undertaken(0). Options: 1: minimum value DATMIN used.; 2:
  minimum DATMIN and maximum DATMAX values used.; 3: matching expression used.
  Range=0,3
  Values=0,1,2,3
  Default=0
  Required=No

* **inrange**:
  Type of in-place update (0). Where the attribute field exists in the input file, only
  those values satisfying the range or pattern will be updated if set to 1. All records
  are output.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('attset')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute attset
print("Running attset...")
dm_cmd.attset(
    in_i='t_assays',  # required
    legend_i='t_input_file',  # required
    out_o='t_attset_out',  # required
    # datflds_f=['optional'],  # optional
    # attribs_f=['optional'],  # optional
    # mode_p=0,  # optional
    # inrange_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("attset execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_attset_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")